# Ecom LLM (Modular + Magic Commands)

This notebook is now a thin orchestrator.
- Core logic is implemented in `ecom_llm.py`
- The module is loaded with notebook magic (`%run`)
- Run cells in order

In this Notebook we run the generator and the judge using the new prompts from Notebook assignment_01_pt2_new_prompt.ipynb

In [1]:
# Optional dependency install
%pip install -q openai pandas tqdm pydantic openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Load all modular functions/classes into notebook scope using magic command
%run ./ecom_llm.py
%run ./llm_judge.py

In [3]:
# Override defaults to run with bigger models in this notebook only
import llm_config
import llm_generator
import llm_judge

llm_generator.GEN_MODEL = "Qwen/Qwen3-30B-A3B-Instruct-2507"
llm_judge.JUDGE_MODEL = "meta-llama/Llama-3.3-70B-Instruct"

# Keep config values aligned for easier debugging/printing
llm_config.GEN_MODEL = llm_generator.GEN_MODEL
llm_config.JUDGE_MODEL = llm_judge.JUDGE_MODEL

print("GEN_MODEL:", llm_generator.GEN_MODEL)
print("JUDGE_MODEL:", llm_judge.JUDGE_MODEL)

GEN_MODEL: Qwen/Qwen3-30B-A3B-Instruct-2507
JUDGE_MODEL: meta-llama/Llama-3.3-70B-Instruct


In [ ]:
import os
import pandas as pd
from IPython.display import HTML, display
import numpy as np
import re
from sklearn.metrics import confusion_matrix, classification_report
from llm_scoring import final_pass_fail

os.environ["NEBIUS_API_KEY"] = "Your_key"

In [6]:
from llm_client import get_client
client = get_client()

g = client.chat.completions.create(
    model=llm_generator.GEN_MODEL,
    messages=[{"role":"user","content":"hi"}],
    max_tokens=1,
    temperature=0
)
j = client.chat.completions.create(
    model=llm_judge.JUDGE_MODEL,
    messages=[{"role":"user","content":"hi"}],
    max_tokens=1,
    temperature=0
)

print("API used for generation:", g.model)
print("API used for judging:", j.model)

API used for generation: Qwen/Qwen3-30B-A3B-Instruct-2507
API used for judging: meta-llama/Llama-3.3-70B-Instruct


In [ ]:
data_path = "data/raw/ecommerce_dataset.csv"

results_df, full_df, summary_overview, summary_distribution = run_assignment(
    df=None,
    excel_path="data/generated/ecommerce_sheet_bigger_model.xlsx",
    data_path=data_path,
)

print("Saved and updated: ecommerce_sheet_bigger_model.xlsx")
print(f"Loaded dataset from: {data_path}")
print("results_df (trimmed output columns):")       
display(results_df.head())
print("full_df (includes explanation columns):")
display(full_df.head())

Task 2: Generating:   0%|          | 0/50 [00:00<?, ?it/s]

Task 5: Judging:   0%|          | 0/50 [00:00<?, ?it/s]

Saved and updated: ecommerce_sheet_bigger_model.xlsx
Loaded dataset from: data/raw/ecommerce_dataset.csv
results_df (trimmed output columns):


,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,tone,length,grounding,latency_rating,cost_rating,final_score
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,You’ll enjoy smoother performance and sharper ...,1757,408,91,good,good,good,good,good,good,good,pass
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Capture every detail with the 200 MP camera an...,2246,413,103,good,good,good,good,good,ok,good,pass
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Capture every moment with stunning clarity and...,2588,424,104,good,good,good,good,good,ok,good,pass
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,You’ll enjoy immersive sound with the Sony WH-...,2440,407,115,good,good,good,good,good,ok,good,pass
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,"You’ll enjoy rich, balanced sound that’s tailo...",1809,397,92,good,good,good,good,good,good,good,pass


full_df (includes explanation columns):


,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,...,length,grounding,latency_rating,cost_rating,final_score,fluency_explanation,grammar_explanation,tone_explanation,length_explanation,grounding_explanation
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,You’ll enjoy smoother performance and sharper ...,1757,408,91,good,good,...,good,good,good,good,pass,"The description reads naturally, with each sen...",The description contains no spelling or punctu...,"The tone is informative and helpful, addressin...",I counted 76 words in the generated descriptio...,Step A: The required facts from the source dat...
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Capture every detail with the 200 MP camera an...,2246,413,103,good,good,...,good,good,ok,good,pass,"The generated description reads naturally, wit...",The description contains no spelling or punctu...,The tone of the description is informative and...,I counted 76 words in the generated descriptio...,Step A: The required facts from the source dat...
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Capture every moment with stunning clarity and...,2588,424,104,good,good,...,good,good,ok,good,pass,"The description reads naturally, with every se...",The description contains zero spelling or punc...,The tone of the description is warm and inviti...,I counted 76 words in the description. This fa...,Step A: The required facts from the source dat...
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,You’ll enjoy immersive sound with the Sony WH-...,2440,407,115,good,good,...,good,good,ok,good,pass,The description is well-structured and easy to...,The description contains no spelling or punctu...,The tone of the description is informative and...,I counted 76 words in the description. This fa...,Step A: The required facts from the source dat...
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,"You’ll enjoy rich, balanced sound that’s tailo...",1809,397,92,good,good,...,good,good,good,good,pass,The description is well-structured and easy to...,The description contains no spelling or punctu...,"The tone is informative and customer-focused, ...",I counted 76 words in the generated descriptio...,Step A: The required facts from the source dat...


In [8]:
# Add cost column per row to both dataframes
# $0.02 / 1M In
# $0.06 / 1M Out (with caching)
results_df["cost"] = results_df.apply(
    lambda row: 0.02 * row["input_tokens"] / 1e6 + 0.06 * row["output_tokens"] / 1e6,
    axis=1
)
full_df["cost"] = full_df.apply(
    lambda row: 0.02 * row["input_tokens"] / 1e6 + 0.06 * row["output_tokens"] / 1e6,
    axis=1
)

In [10]:
results_df.to_excel("data/generated/assignment_01_bigger_model.xlsx", index=False)
full_df.to_excel("data/generated/assignment_01_full_bigger_model.xlsx", index=False)
print("Saved: data/generated/assignment_01_bigger_model.xlsx and data/generated/assignment_01_full_new.xlsx")

Saved: data/generated/assignment_01_bigger_model.xlsx and data/generated/assignment_01_full_new.xlsx


In [11]:
rows = results_df.sample(n=15, random_state=42).copy()
rows

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,tone,length,grounding,latency_rating,cost_rating,final_score,cost
13,Nintendo Switch OLED,"features: 7″ OLED screen, docked & handheld mo...",plastic chassis,1‑year limited warranty,You’ll enjoy vibrant colors and crisp detail o...,1611,398,76,good,good,good,good,good,good,good,pass,0.000013
39,Blink Outdoor 4,"features: 1080p HD, 2‑year battery, weather‑re...",plastic,1‑year limited warranty,Capture every moment with crystal-clear 1080p ...,1415,396,93,good,good,ok,good,good,good,good,pass,0.000013
30,LEGO Star Wars Millennium Falcon 75192,"features: 7 541 pieces, detailed interior, min...",ABS plastic bricks,lifetime replacement of missing bricks,Build your own epic Star Wars adventure with t...,1698,405,126,good,good,good,good,good,good,good,pass,0.000016
45,SanDisk Extreme PRO 128 GB SDXC,"features: UHS‑I U3, 200 MB/s read, waterproof;...",plastic,lifetime limited warranty,You capture every breathtaking moment with spe...,2040,405,111,good,good,good,good,good,ok,good,pass,0.000015
17,Keurig K‑Supreme Plus Smart,"features: multistream extraction, BrewID, 78 o...",brushed stainless steel accents,1‑year limited warranty,"You enjoy rich, restaurant-quality coffee with...",1531,401,95,good,good,good,good,good,good,good,pass,0.000014
48,BenQ PD2725U 27″ Monitor,"features: 4K UHD IPS, 98% P3, Thunderbolt 3; c...",metal stand,3‑year warranty,"You’ll enjoy vibrant, accurate colors and stun...",1587,408,97,good,good,good,good,good,good,good,pass,0.000014
26,Adidas Ultraboost Light,"features: Light BOOST midsole, Primeknit+ uppe...",textile & synthetic,2‑year manufacturer warranty,You’ll feel every step light and responsive wi...,1356,418,101,good,good,good,good,good,good,good,pass,0.000014
25,Nike Air Zoom Pegasus 40,"features: React foam, Zoom Air units, engineer...",textile & synthetic,2‑year manufacturer warranty,You’ll feel every step light and responsive wi...,1824,394,111,good,good,good,good,good,good,good,pass,0.000015
32,Kindle Paperwhite 11th Gen,"features: 6.8″ glare‑free display, 10‑week bat...",recycled plastic,1‑year limited warranty,You’ll enjoy hours of reading with the Kindle ...,5630,404,84,good,good,good,good,good,bad,good,fail,0.000013
19,Dyson V15 Detect Vacuum,"features: laser dust detection, HEPA filtratio...",polycarbonate & aluminum,2‑year parts & labor,You’ll capture every speck of dust with the Dy...,1643,402,115,good,good,ok,good,good,good,good,pass,0.000015


In [12]:
import re

def count_words(text: str) -> int:
    # Strip punctuation artifacts, split on whitespace
    return len(re.findall(r'\b\w+\b', text))

def classify_length(word_count: int) -> str:
    if 50 <= word_count <= 90:
        return "good"
    elif (40 <= word_count <= 49) or (91 <= word_count <= 110):
        return "ok"
    else:
        return "bad"

In [13]:
rows["generated_description"] = rows["generated_description"].str.replace("\n", "<br>", regex=False)
from IPython.display import HTML, display
display(HTML(rows[["product_name","generated_description", "Product_attribute_list"]].to_html(escape=False, index=False)))

product_name,generated_description,Product_attribute_list
Nintendo Switch OLED,"You’ll enjoy vibrant colors and crisp detail on the 7″ OLED screen, whether you’re playing handheld or docked. With 64 GB of storage, you’ve got room for all your favorite games. The durable plastic chassis feels solid in your hands, and you’re protected with a 1-year limited warranty. Play your way, anytime, anywhere.","features: 7″ OLED screen, docked & handheld modes, 64 GB storage; capacity: large"
Blink Outdoor 4,"Capture every moment with crystal-clear 1080p HD video, even in the harshest weather. Your Blink Outdoor 4 is built to last, with a weather-resistant design and a 2-year battery that keeps you protected without constant maintenance. The sleek plastic construction ensures durability, while the limited edition style adds a touch of uniqueness. Backed by a 1-year limited warranty, you can enjoy peace of mind knowing your security is covered.","features: 1080p HD, 2‑year battery, weather‑resistant; limited edition"
LEGO Star Wars Millennium Falcon 75192,"Build your own epic Star Wars adventure with the LEGO Star Wars Millennium Falcon 75192, featuring 7 541 detailed pieces and a fully equipped interior. You’ll enjoy assembling each intricate panel and placing your favorite minifigs inside, from Han Solo to Chewbacca. The long-lasting battery powers the working lights and sound effects, bringing the ship to life. Made with durable ABS plastic bricks, it’s built to last. Should any brick go missing, we’ll replace it for life—no questions asked. You’ve got a ship, you’ve got a story, now build it.","features: 7 541 pieces, detailed interior, minifigs included; battery: long‑lasting"
SanDisk Extreme PRO 128 GB SDXC,"You capture every breathtaking moment with speed and confidence using the SanDisk Extreme PRO 128 GB SDXC card. Designed for pros and enthusiasts, it reads at 200 MB/s, ensuring smooth 4K video recording and fast file transfers. Its waterproof construction protects your memories in tough conditions, while the long-lasting battery compatibility keeps your camera running. Built with durable plastic, it’s ready for action wherever your creativity takes you. Backed by a lifetime limited warranty, it’s a dependable companion for every adventure.","features: UHS‑I U3, 200 MB/s read, waterproof; battery: long‑lasting"
Keurig K‑Supreme Plus Smart,"You enjoy rich, restaurant-quality coffee with every brew, thanks to multistream extraction that ensures balanced flavor. BrewID technology remembers your favorite drinks, so your perfect cup is ready in seconds. The large 78 oz reservoir means fewer refills, keeping your mornings smooth. With brushed stainless steel accents, your Keurig K-Supreme Plus Smart adds a sleek touch to any kitchen. It’s built to last, backed by a 1-year limited warranty.","features: multistream extraction, BrewID, 78 oz reservoir; capacity: large"
BenQ PD2725U 27″ Monitor,"You’ll enjoy vibrant, accurate colors and stunning detail on your BenQ PD2725U 27″ Monitor, thanks to its 4K UHD IPS panel and 98% P3 color coverage. Connect seamlessly with Thunderbolt 3 for fast data transfer and daisy-chaining. The sleek metal stand supports stable, elegant positioning. With a 3-year warranty, your investment is protected. Your workflow meets the highest standards of clarity and reliability.","features: 4K UHD IPS, 98% P3, Thunderbolt 3; color options: multiple"
Adidas Ultraboost Light,"You’ll feel every step light and responsive with the Adidas Ultraboost Light, featuring a Light BOOST midsole that cushions your stride. The Primeknit+ upper wraps your foot in breathable comfort, while Continental rubber delivers lasting grip on any surface. Crafted from durable textile and synthetic materials, these shoes are built to keep up with your pace. With a 2-year manufacturer warranty and a 4.7/5 rating from wearers, you’re choosing a trusted companion for every journey.","features: Light BOOST midsole, Primeknit+ upper, Continental ru

**The results are like night and day and the better judge agrees with it**. Seems that with a better generator (30b) and a better judge (70b) we can get impressive results.